In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


In [7]:
## Configuration

## Path of data file
path = "D:/Columbia/MonteCarlo/TermProject/data/"

## Name of data file
file_name = ["us_equity_adj_close.csv", "us_bond_intermediate_core_adj_close.csv"]

## Date
start_date = "2005-12-31"
end_date = "2025-12-31"



In [8]:
## Equity data
df_equity = pd.read_csv(path + file_name[0], index_col=0, parse_dates=True)
df_equity.index = pd.to_datetime(df_equity.index)
df_equity.index.name = "as_of"
df_equity = df_equity.loc[start_date:end_date]

df_equity = df_equity.reset_index(drop=True)
df_equity = df_equity.dropna(axis=1, how="any")

df_equity["time_index"] = df_equity.index

print("Equity data shape:")
print(df_equity.shape)
print(df_equity.head())


## Bond data
df_bond = pd.read_csv(path + file_name[1], index_col=0, parse_dates=True)
df_bond.index = pd.to_datetime(df_bond.index)
df_bond.index.name = "as_of"
df_bond = df_bond.loc[start_date:end_date]

## df_bond = df_bond.dropna(axis=1, how="any")

print("Bond data shape:")
print(df_bond.shape)



Equity data shape:
(6941, 27)
    B13779   B09156   B06011   B13582   B08422   B08466   B05011   B01060  \
0  19.9535  43.4471  30.1593  14.0244  42.8652  131.657  53.5292  16.9594   
1  19.9535  43.4471  30.1593  14.0244  42.8652  131.657  53.5292  16.9594   
2  19.9535  43.4471  30.1593  14.0244  42.8652  131.657  53.5292  16.9594   
3  20.2659  44.1489  30.6375  14.2558  43.6358  133.624  53.9982  17.2677   
4  20.3310  44.3865  30.7272  14.3363  43.8766  134.849  54.2483  17.2883   

    B11079   B12752  ...   B13661   B09934  B06127   B02943   B03459   B06905  \
0  10.8024  22.8143  ...  28.8240  61.2189    9.24  257.068  340.075  88.7742   
1  10.8024  22.8143  ...  28.8240  61.2189    9.24  257.068  340.075  88.7742   
2  10.8024  22.8143  ...  28.8240  61.2189    9.24  257.068  340.075  88.7742   
3  10.9134  23.3023  ...  29.2396  62.1187    9.41  261.788  343.819  91.0168   
4  11.0043  23.4541  ...  29.3510  62.4936    9.49  263.676  344.800  91.6354   

    B15668   B05924 

In [9]:
## Parameters
Start_index = 0
End_index = len(df_equity) - 1
Tau = 252
Delta = 21

print("Start index:", Start_index)
print("End index:", End_index)

Start index: 0
End index: 6940


In [10]:
def calculate_L(s_index, e_index, delta, tau) -> int:
    """
    Calculate L = floor((e_index - tau + 1 ) / delta)

    Parameters
    s_index (int): Start index of the data
    e_index (int): End index of the data
    delta (int): Time step size
    tau (int): Time horizon
    """

    L = np.floor((e_index - s_index + 1 - tau + 1) / delta)
    return L

In [11]:
def single_historical_distribution(symbol, s_index, e_index, delta, tau):
    """
    Calculate the historical distribution of returns for a given symbol and time horizon.

    Parameters:
    symbol (str): The symbol of the asset.
    s_index (int): Start index of the data.
    e_index (int): End index of the data.
    delta (int): Time step size.
    tau (int): The time horizon in days.

    Returns:
    pandas.DataFrame: DataFrame with index and returns columns.
    pandas.DataFrame: The DataFrame of historical distribution of returns with time indices.
    """
    # Get the price data for the specified symbol
    if symbol in df_equity.columns:
        price_data = df_equity[symbol]
    elif symbol in df_bond.columns:
        price_data = df_bond[symbol]
    else:
        raise ValueError(f"Symbol {symbol} not found in either equity or bond data.")

    # Create a DataFrame
    df = pd.DataFrame()
    df["time_index"] = range(s_index, e_index + 1)

    # Calculate returns
    returns = (price_data / price_data.shift(tau) - 1)
    df["returns"] = returns

    # Calculate L
    L = calculate_L(s_index, e_index, delta, tau)

    distribution_df = pd.DataFrame()
    dis_time_indices = [1+ s_index + i * delta for i in range(int(L))]
    distribution_df["time_index"] = dis_time_indices
    distribution_df["returns"] = returns[dis_time_indices].values

    return df, distribution_df

In [13]:
## Test
df_test, distribution_test = single_historical_distribution("B13779", Start_index, End_index, Delta, Tau)
print("Test data shape:")
print(df_test.shape)
print(df_test.tail())
print("Distribution data shape:")
print(distribution_test.shape)
print(distribution_test)


Test data shape:
(6941, 2)
      time_index   returns
6936        6936  0.173031
6937        6937  0.173031
6938        6938  0.173031
6939        6939  0.152666
6940        6940  0.136004
Distribution data shape:
(318, 2)
     time_index   returns
0             1       NaN
1            22       NaN
2            43       NaN
3            64       NaN
4            85       NaN
..          ...       ...
313        6574  0.155609
314        6595  0.173346
315        6616  0.172255
316        6637  0.187501
317        6658  0.163948

[318 rows x 2 columns]
